In [ ]:
import tensorflow as tf
import tensorflow.keras as keras
import numpy as np
from tensorflow.keras import *
import matplotlib.pyplot as plt
import pandas

In [ ]:
dataset = pandas.read_csv(r'/kaggle/input/az-handwritten-alphabets-in-csv-format/A_Z Handwritten Data.csv')
dataset

In [ ]:
X_train = dataset.iloc[:, 1:].values.reshape(-1, 28, 28, 1) / 255.0
y_train = dataset.iloc[:, 0].values
img_size = 28
X_train.shape, y_train.shape

In [ ]:
def show_images(images, img_size = img_size, n_channels = 1):
    plt.figure(figsize = (8, 8))
    grid = images[:32].reshape(4, 8, img_size, img_size, n_channels)
    grid = tf.transpose(grid, [0, 2, 1, 3, 4])
    grid = tf.reshape(grid, [4 * img_size, 8 * img_size, n_channels])
    plt.imshow(grid)
    plt.xticks([])
    plt.yticks([])
    plt.show()
show_images(X_train, img_size)

In [ ]:
class CVAE(keras.Model):
    def __init__(self, latent_dim = 32, num_classes = 47, KL_coef = 0.5):
        super().__init__()
        self.latent_dim = latent_dim
        self.KL_coef = KL_coef
        self.encoder = keras.Sequential([
            layers.Conv2D(32, kernel_size = (3, 3), strides = 1, padding = 'same'),
            layers.Dropout(0.2),
            layers.LeakyReLU(0.3),
            layers.BatchNormalization(),
            
            layers.Conv2D(64, kernel_size = (3, 3), strides = 1, padding = 'same'),
            layers.Dropout(0.2),
            layers.LeakyReLU(0.3),
            layers.BatchNormalization(),
            
            layers.Conv2D(128, kernel_size = (3, 3), strides = 2, padding = 'same'),
            layers.Dropout(0.2),
            layers.LeakyReLU(0.3),
            layers.BatchNormalization(),
            
            layers.Conv2D(256, kernel_size = (3, 3), strides = 2, padding = 'same'),
            layers.Dropout(0.2),
            layers.LeakyReLU(0.3),
            layers.BatchNormalization(),
            
            layers.Flatten()
        ])
        self.embedding_layer = layers.Embedding(num_classes, self.latent_dim)
        self.get_mean = layers.Dense(self.latent_dim, kernel_initializer = keras.initializers.Zeros())
        self.get_logvar = layers.Dense(self.latent_dim, kernel_initializer = keras.initializers.Zeros())
        self.decoder = keras.Sequential([
            layers.Dense(256 * 7 * 7),
            layers.Dropout(0.2),
            layers.BatchNormalization(),
            layers.Reshape((7, 7, 256)),
            
            layers.Conv2DTranspose(128, kernel_size = (3, 3), strides = 2, padding = 'same'),
            layers.Dropout(0.2),
            layers.LeakyReLU(0.3),
            layers.BatchNormalization(),

            layers.Conv2DTranspose(64, kernel_size = (3, 3), strides = 2, padding = 'same'),
            layers.Dropout(0.2),
            layers.LeakyReLU(0.3),
            layers.BatchNormalization(),
            
            layers.Conv2DTranspose(32, kernel_size = (3, 3), strides = 1, padding = 'same'),
            layers.Dropout(0.2),
            layers.LeakyReLU(0.3),
            layers.BatchNormalization(),
            
            layers.Conv2DTranspose(1, kernel_size = (3, 3), strides = 1, padding = 'same', activation = 'sigmoid'),
        ])
    
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, x):
        return self.decoder(x)
    
    def sample(self, mean, logvar):
        epsilon = tf.random.normal(tf.shape(mean))
        return mean + tf.exp(0.5 * logvar) * epsilon
    
    def call(self, batch):
        images, labels = batch
        encoded = self.encode(images)
        mean, logvar = self.get_mean(encoded), self.get_logvar(encoded)
        latent_space = self.sample(mean, logvar)
        embeddings = self.embedding_layer(labels)
        z = latent_space + embeddings
        return mean, logvar, self.decode(z)
    
    def generate(self, labels):
        noise = tf.random.normal((tf.shape(labels)[0], self.latent_dim))
        embeddings = self.embedding_layer(labels)
        z = noise + embeddings
        return np.array(self.decoder(z))
    
    def vae_loss(self, inputs, outputs, z_mean, z_log_var):
        reconstruction_loss = tf.keras.losses.MeanSquaredError()(inputs, outputs)
        kl_loss = -self.KL_coef * tf.reduce_mean(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
        total_loss = reconstruction_loss + kl_loss
        return kl_loss, reconstruction_loss, total_loss

    
    def train_step(self, batch):
        real_images, labels = batch
        with tf.GradientTape() as tape:
            mean, logvar, y_pred = self.call(batch)
            kl_loss, reconstruction_loss, loss = self.vae_loss(real_images, y_pred, mean, logvar)
        gradients = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))
        
        return {'kl_loss' : kl_loss, 'reconstruction_loss': reconstruction_loss, 'total_loss' : loss}

In [ ]:
model = CVAE(1024, 27)
model.compile(optimizer = 'adam')

In [ ]:
model.fit(X_train, y_train, epochs = 100)

In [ ]:
a1 = model.generate(np.array([14, 12, 0, 17, 8, 14, 21, 8, 2]))
plt.imshow(np.concatenate([a1[i] for i in range(len(a1))], axis = 1))

In [ ]:
a1 = model.generate(np.array([0, 7, 12, 4, 3]))
a2 = model.generate(np.array([7, 0, 13, 24]))
plt.imshow(np.concatenate([a1[i] for i in range(len(a1))] + [a2[i] for i in range(len(a2))], axis = 1))

In [ ]:
a1 = model.generate(np.array([18, 0, 7, 4, 17]))
plt.imshow(np.concatenate([a1[i] for i in range(len(a1))], axis = 1))

In [ ]:
a2 = model.generate(np.array([18, 14, 0, 0, 3]))
plt.imshow(np.concatenate([a2[i] for i in range(len(a2))], axis = 1))

In [ ]:
a3 = model.generate(np.array([0, 25, 8, 25, 0]))
plt.imshow(np.concatenate([a3[i] for i in range(len(a2))], axis = 1))

In [ ]:
model.save_weights('VAE_TEXT_GEN.weights.h5')